# ML Dataset Audit: Normal vs LOCA Early Warning

This audit checks the local NPPAD release before training a new classifier. It covers the available normal trajectories, LOCA transient-report events, README-backed feature roles, label windows, group-only split design, and leakage risks.

The target research setting is **LOCA Early Detection Before Protection-System Actuation**. This notebook does not train a final model and does not treat the result as a safety claim.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path(r'C:/Users/18205/NPP-Guard')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_loader import DEFAULT_LOCA_DIR, DEFAULT_NORMAL_FILE
from features import (
    build_loca_event_table,
    candidate_pre_protection_features,
    excluded_pre_protection_features,
    feature_group_table,
    interleaved_group_split,
    label_time_windows,
)

DATA_ROOT = PROJECT_ROOT / 'data' / 'NuclearPowerPlantAccidentData'
OPERATION_ROOT = DATA_ROOT / 'Operation_csv_data'
VARIABLE_ROOT = DATA_ROOT / 'Variable_Power_Data'
RESULTS_ROOT = PROJECT_ROOT / 'results'
RESULTS_ROOT.mkdir(exist_ok=True)
LOCA_FILES = sorted(DEFAULT_LOCA_DIR.glob('*.csv'), key=lambda path: int(path.stem))
assert len(LOCA_FILES) == 100
print('Project:', PROJECT_ROOT)
print('LOCA trajectories:', len(LOCA_FILES))


Project: C:\Users\18205\NPP-Guard
LOCA trajectories: 100


## 1. Available data and normal trajectories

The local NPPAD README defines the original operating-condition classes in Table 1 and the 97 operation columns in Table 2. Fixed-power Normal is checked separately from the variable-power extension because a power-transition normal run is not the same operating condition as a fixed full-power LOCA baseline.


In [2]:
inventory_rows = []
for directory in sorted(path for path in OPERATION_ROOT.iterdir() if path.is_dir()):
    files = sorted(directory.glob('*.csv'))
    if not files:
        continue
    sample = pd.read_csv(files[0], nrows=1)
    inventory_rows.append({
        'class': directory.name,
        'csv_trajectories': len(files),
        'columns_in_first_csv': len(sample.columns),
        'first_file': files[0].name,
    })
fixed_inventory = pd.DataFrame(inventory_rows)
display(fixed_inventory)

manifest = pd.read_csv(VARIABLE_ROOT / 'manifest.csv')
normal_manifest = manifest.loc[manifest['category'].eq('normal_power_transition')].copy()
normal_manifest['mdb_exists'] = normal_manifest.apply(
    lambda row: (DATA_ROOT / row['relative_path'] / row['operation_file']).exists(), axis=1
)
print('Variable-power normal scenarios:', len(normal_manifest))
display(normal_manifest[['scenario_id', 'start_power_pct', 'end_power_pct', 'mdb_exists']])

try:
    import pyodbc
    access_driver = 'Microsoft Access Driver (*.mdb, *.accdb)' in pyodbc.drivers()
except ImportError:
    pyodbc = None
    access_driver = False

def mdb_plotdata_shape(path):
    if not access_driver:
        return {'mdb_rows': None, 'mdb_columns': None}
    connection = pyodbc.connect(
        r'Driver={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=' + str(path.resolve()) + ';'
    )
    cursor = connection.cursor()
    cursor.execute('SELECT COUNT(*) FROM PlotData')
    rows = int(cursor.fetchone()[0])
    cursor.execute('SELECT TOP 1 * FROM PlotData')
    columns = len(cursor.description)
    connection.close()
    return {'mdb_rows': rows, 'mdb_columns': columns}

if access_driver:
    normal_manifest[['mdb_rows', 'mdb_columns']] = normal_manifest.apply(
        lambda row: pd.Series(mdb_plotdata_shape(DATA_ROOT / row['relative_path'] / row['operation_file'])),
        axis=1,
    )
display(normal_manifest[['scenario_id', 'mdb_rows', 'mdb_columns']])
fixed_normal_count = int(fixed_inventory.loc[fixed_inventory['class'].eq('Normal'), 'csv_trajectories'].iloc[0])
print('Fixed-power Normal CSV trajectories:', fixed_normal_count)
print('Variable-power Normal MDB trajectories:', len(normal_manifest))


,class,csv_trajectories,columns_in_first_csv,first_file
0,ATWS,1,97,1.csv
1,FLB,100,97,1.csv
2,LACP,1,97,1.csv
3,LLB,101,97,1.csv
4,LOCA,100,97,1.csv
5,LOCAC,100,97,1.csv
6,LOF,1,97,1.csv
7,LR,99,97,1.csv
8,MD,100,97,1.csv
9,Normal,1,97,1.csv


Variable-power normal scenarios: 4


,scenario_id,start_power_pct,end_power_pct,mdb_exists
0,NORM_80_to_100,80,100,True
1,NORM_100_to_80,100,80,True
2,NORM_70_to_90,70,90,True
3,NORM_90_to_70,90,70,True


,scenario_id,mdb_rows,mdb_columns
0,NORM_80_to_100,228,97
1,NORM_100_to_80,375,97
2,NORM_70_to_90,258,97
3,NORM_90_to_70,287,97


Fixed-power Normal CSV trajectories: 1
Variable-power Normal MDB trajectories: 4


## 2. LOCA event table

Each LOCA report is parsed by event description, not by severity-derived assumptions. first_protection_s is the earlier of the reported HPI pump-1 actuation and Reactor Scram. Missing secondary events remain missing rather than being imputed.


In [3]:
events = build_loca_event_table()

def first_positive_sensor(path, column):
    frame = pd.read_csv(path, usecols=['TIME', column])
    hits = frame.loc[pd.to_numeric(frame[column], errors='coerce').gt(1e-9), 'TIME']
    return float(hits.iloc[0]) if len(hits) else np.nan

events['whpi_first_positive_s'] = [
    first_positive_sensor(OPERATION_ROOT / 'LOCA' / f'{severity}.csv', 'WHPI')
    for severity in events['severity']
]
events['report_hpi_minus_whpi_s'] = events['hpi_pump_1_s'] - events['whpi_first_positive_s']
events.to_csv(RESULTS_ROOT / 'loca_event_table.csv', index=False)

display(events[['sample_id', 'severity', 'injection_s', 'hpi_pump_1_s', 'scram_s', 'first_protection_s']].head())
print('Injection times:', events['injection_s'].nunique(), 'unique value(s)', events['injection_s'].unique())
print('First protection range:', events['first_protection_s'].min(), 'to', events['first_protection_s'].max(), 's')
print('First protection median:', events['first_protection_s'].median(), 's')
print('HPI before Scram:', int(events['hpi_pump_1_s'].lt(events['scram_s']).sum()))
print('Scram before HPI:', int(events['scram_s'].lt(events['hpi_pump_1_s']).sum()))
print('Severity/event correlation:', round(events[['severity', 'first_protection_s']].corr().iloc[0, 1], 3))


,sample_id,severity,injection_s,hpi_pump_1_s,scram_s,first_protection_s
0,LOCA_001,1,0.5,2031.5,2032.5,2031.5
1,LOCA_002,2,0.5,1094.5,1095.5,1094.5
2,LOCA_003,3,0.5,790.5,791.5,790.5
3,LOCA_004,4,0.5,577.0,569.0,569.0
4,LOCA_005,5,0.5,427.0,417.5,417.5


Injection times: 1 unique value(s) [0.5]
First protection range: 45.0 to 2031.5 s
First protection median: 126.75 s
HPI before Scram: 3
Scram before HPI: 97
Severity/event correlation: -0.416


## 3. Window definitions and Normal/LOCA alignment

The strict labels are: normal_or_pre_fault for a control trajectory or a pre-injection calibration point, early_loca for injection_s < TIME < first_protection_s, and post_protection_loca for TIME >= first_protection_s. The first LOCA positive observation is normally the 10 s sample because the CSV grid is 10 s and the reports place injection at 0.5 s.


In [4]:
window_rows = []
for path in LOCA_FILES:
    severity = int(path.stem)
    event = events.loc[events['severity'].eq(severity)].iloc[0]
    frame = pd.read_csv(path, usecols=['TIME'])
    labels = label_time_windows(frame['TIME'], event['injection_s'], event['first_protection_s'])
    counts = labels.value_counts()
    window_rows.append({
        'sample_id': event['sample_id'],
        'severity': severity,
        'early_loca_rows': int(counts.get('early_loca', 0)),
        'post_protection_rows': int(counts.get('post_protection_loca', 0)),
        'pre_fault_rows': int(counts.get('normal_or_pre_fault', 0)),
    })
window_counts = pd.DataFrame(window_rows)
display(window_counts.describe())

normal = pd.read_csv(DEFAULT_NORMAL_FILE)
alignment_rows = []
for time_s in [0, 10, 100]:
    normal_row = normal.loc[normal['TIME'].eq(time_s)].iloc[0]
    differences = []
    for path in LOCA_FILES:
        loca_row = pd.read_csv(path).loc[lambda frame: frame['TIME'].eq(time_s)].iloc[0]
        differences.append(float((loca_row.drop('TIME') - normal_row.drop('TIME')).abs().max()))
    alignment_rows.append({
        'time_s': time_s,
        'max_case_abs_difference': max(differences),
        'median_case_abs_difference': float(np.median(differences)),
    })
alignment = pd.DataFrame(alignment_rows)
display(alignment)

time_counts = pd.concat(
    [pd.read_csv(path, usecols=['TIME'])['TIME'] for path in LOCA_FILES],
    ignore_index=True,
).value_counts()
shared_time_points = int((time_counts > 1).sum())
print('LOCA time points shared by multiple trajectories:', shared_time_points)
print('Early LOCA rows per case:', int(window_counts['early_loca_rows'].min()), 'to', int(window_counts['early_loca_rows'].max()))
print('Normal trajectory rows:', len(normal), 'end time:', float(normal['TIME'].iloc[-1]))


,severity,early_loca_rows,post_protection_rows,pre_fault_rows
count,100.000000,100.000000,100.000000,100.0
mean,50.500000,17.610000,371.560000,1.0
std,29.011492,22.845504,41.822695,0.0
min,1.000000,4.000000,271.000000,1.0
25%,25.750000,12.000000,340.500000,1.0
50%,50.500000,12.000000,372.000000,1.0
75%,75.250000,14.000000,400.000000,1.0
max,100.000000,203.000000,453.000000,1.0


,time_s,max_case_abs_difference,median_case_abs_difference
0,0,0.000000,0.000000
1,10,210.301636,210.301636
2,100,635.524658,635.524658


LOCA time points shared by multiple trajectories: 471
Early LOCA rows per case: 4 to 203
Normal trajectory rows: 302 end time: 3010.0


## 4. README-backed feature grouping

The grouping uses the local NPPAD README Table 2 definitions. It does not infer meaning from abbreviations alone. The strict first baseline admits only observable_process columns. Direct break/leak quantities, action/control variables, radiological/dose variables, and consequence or margin variables are excluded or reserved for sensitivity studies.


In [5]:
operation_columns = pd.read_csv(LOCA_FILES[0], nrows=0).columns.tolist()
feature_groups = feature_group_table(operation_columns)
assert len(operation_columns) == 97
assert len(feature_groups) == 97
assert feature_groups['feature'].tolist() == operation_columns
feature_groups.to_csv(RESULTS_ROOT / 'feature_groups.csv', index=False)
display(feature_groups.groupby(['group', 'ml_policy']).size().rename('count').reset_index())
print('Strict process-only candidate count:', len(candidate_pre_protection_features()))
print('Direct/near-direct exclusion:', feature_groups.loc[feature_groups['group'].eq('direct_accident_or_leak'), 'feature'].tolist())


,group,ml_policy,count
0,direct_accident_or_leak,exclude,8
1,observable_process,candidate,38
2,protection_or_control_action,exclude,18
3,radiological_or_dose,exclude,13
4,safety_margin_or_consequence,exclude,10
5,safety_margin_or_consequence,review,9
6,time_index,exclude,1


Strict process-only candidate count: 38
Direct/near-direct exclusion: ['WLR', 'HLW', 'WTRA', 'WTRB', 'WBK', 'MBK', 'EBK', 'WFLB']


## 5. Recommended labels and group split

Recommended first experiment:

- Normal: independent fixed-power normal trajectories on the same observation grid and operating condition. Use the four variable-power normal MDB cases only in a separate power-transition study.
- Early LOCA: samples strictly after the reported injection and strictly before the first HPI/Scram protection action for the same trajectory.
- Post-protection LOCA: samples at or after the first protection action. Keep for descriptive analysis and a separate post-protection experiment; do not mix into the early-warning training target.
- Do not use severity or sample_id as model features. Severity is the LOCA file number and is coupled to the accident setting; retain it only for stratification and reporting.
- Split by complete trajectory group, never by row. The following deterministic template interleaves LOCA severities to avoid putting one severity range entirely in one split; it is a split audit, not a claim that the current Normal-vs-LOCA dataset is ready.


In [6]:
split_plan = interleaved_group_split(events['sample_id'].tolist())
split_sets = {name: set(split_plan.loc[split_plan['split'].eq(name), 'group_id']) for name in ['train', 'validation', 'test']}
assert split_sets['train'].isdisjoint(split_sets['validation'])
assert split_sets['train'].isdisjoint(split_sets['test'])
assert split_sets['validation'].isdisjoint(split_sets['test'])
assert len(set.union(*split_sets.values())) == 100
display(split_plan.groupby('split').size().rename('trajectory_count'))

audit_rows = [
    {'check': 'same_sample_id_across_splits', 'status': 'pass_for_template', 'evidence': 'Each of 100 LOCA trajectories is assigned to exactly one split.', 'action': 'Keep sample_id as the grouping unit.'},
    {'check': 'row_random_split', 'status': 'high_risk', 'evidence': f'{shared_time_points} time coordinates are shared across LOCA trajectories.', 'action': 'Do not use random row train_test_split.'},
    {'check': 'direct_label_variables', 'status': 'high_risk', 'evidence': 'Break/leak columns are identified in the README-backed feature table.', 'action': 'Exclude the direct_accident_or_leak group.'},
    {'check': 'post_protection_observations', 'status': 'high_risk', 'evidence': 'The report table supplies a first protection time for every LOCA case.', 'action': 'Use injection_s < TIME < first_protection_s for Early LOCA.'},
    {'check': 'severity_label_coupling', 'status': 'high_risk', 'evidence': 'The 1..100 LOCA filename is the README-defined break severity setting.', 'action': 'Use severity only for stratification/reporting.'},
    {'check': 'normal_sample_size', 'status': 'blocker', 'evidence': 'There is one fixed-power Normal CSV trajectory; four other Normal cases are variable-power MDB scenarios.', 'action': 'Acquire independent matched fixed-power Normal trajectories.'},
    {'check': 'normal_loca_condition_match', 'status': 'review', 'evidence': 'Normal and LOCA match at TIME=0 but diverge by TIME=10 and more by TIME=100.', 'action': 'Quantify operating-condition confounding before binary training.'},
]
audit_summary = pd.DataFrame(audit_rows)
audit_summary.to_csv(RESULTS_ROOT / 'ml_dataset_audit_summary.csv', index=False)
display(audit_summary)


split
test          10
train         80
validation    10
Name: trajectory_count, dtype: int64

,check,status,evidence,action
0,same_sample_id_across_splits,pass_for_template,Each of 100 LOCA trajectories is assigned to e...,Keep sample_id as the grouping unit.
1,row_random_split,high_risk,471 time coordinates are shared across LOCA tr...,Do not use random row train_test_split.
2,direct_label_variables,high_risk,Break/leak columns are identified in the READM...,Exclude the direct_accident_or_leak group.
3,post_protection_observations,high_risk,The report table supplies a first protection t...,Use injection_s < TIME < first_protection_s fo...
4,severity_label_coupling,high_risk,The 1..100 LOCA filename is the README-defined...,Use severity only for stratification/reporting.
5,normal_sample_size,blocker,There is one fixed-power Normal CSV trajectory...,Acquire independent matched fixed-power Normal...
6,normal_loca_condition_match,review,Normal and LOCA match at TIME=0 but diverge by...,Quantify operating-condition confounding befor...


## 6. Audit conclusion

**Current status: not ready for a defensible Normal-vs-LOCA Random Forest baseline.** The LOCA event metadata and the strict before-protection window are usable, and a group-only LOCA split can be constructed. The blocking issue is the fixed-power Normal class: one trajectory cannot provide independent Normal train/validation/test groups, and the available Normal/LOCA paths are not demonstrably matched after TIME=0.

The safest next step is to add independent matched fixed-power Normal runs, keep variable-power normals in a separate condition-specific dataset, then train only on Normal and Early LOCA windows with group-based splits. A one-class or reference-based anomaly detector can be studied now, but it should not be reported as a supervised binary classifier baseline.


In [7]:
audit_payload = {
    'ready_for_random_forest_baseline': False,
    'fixed_power_normal_trajectories': fixed_normal_count,
    'variable_power_normal_trajectories': int(len(normal_manifest)),
    'loca_trajectories': int(len(events)),
    'unique_injection_times': int(events['injection_s'].nunique()),
    'injection_time_s': float(events['injection_s'].iloc[0]),
    'first_protection_min_s': float(events['first_protection_s'].min()),
    'first_protection_median_s': float(events['first_protection_s'].median()),
    'first_protection_max_s': float(events['first_protection_s'].max()),
    'strict_candidate_feature_count': int(len(candidate_pre_protection_features())),
    'excluded_feature_count': int(len(excluded_pre_protection_features())),
    'event_table_path': 'results/loca_event_table.csv',
    'feature_table_path': 'results/feature_groups.csv',
    'audit_table_path': 'results/ml_dataset_audit_summary.csv',
}
(RESULTS_ROOT / 'ml_dataset_audit_summary.json').write_text(json.dumps(audit_payload, indent=2), encoding='utf-8')
assert audit_payload['loca_trajectories'] == 100
assert audit_payload['unique_injection_times'] == 1
assert audit_payload['fixed_power_normal_trajectories'] == 1
print(json.dumps(audit_payload, indent=2))


{
  "ready_for_random_forest_baseline": false,
  "fixed_power_normal_trajectories": 1,
  "variable_power_normal_trajectories": 4,
  "loca_trajectories": 100,
  "unique_injection_times": 1,
  "injection_time_s": 0.5,
  "first_protection_min_s": 45.0,
  "first_protection_median_s": 126.75,
  "first_protection_max_s": 2031.5,
  "strict_candidate_feature_count": 38,
  "excluded_feature_count": 59,
  "event_table_path": "results/loca_event_table.csv",
  "feature_table_path": "results/feature_groups.csv",
  "audit_table_path": "results/ml_dataset_audit_summary.csv"
}
